In [1]:
import requests
import pandas as pd


def fetch_live_features():

    # ---------------------------
    # GOES Electron Flux
    # ---------------------------
    goes_url = (
        "https://services.swpc.noaa.gov/"
        "json/goes/primary/integral-electrons-1-day.json"
    )

    goes = pd.DataFrame(
        requests.get(goes_url).json()
    )

    goes["time_tag"] = pd.to_datetime(
        goes["time_tag"]
    )

    goes = goes.sort_values("time_tag")

    flux_1 = float(goes["flux"].iloc[-2])
    flux_3 = float(goes["flux"].iloc[-4])
    flux_6 = float(goes["flux"].iloc[-7])
    flux_12 = float(goes["flux"].iloc[-13])

    # ---------------------------
    # Plasma Data
    # ---------------------------
    plasma_url = (
        "https://services.swpc.noaa.gov/"
        "products/solar-wind/plasma-1-day.json"
    )

    plasma_raw = requests.get(
        plasma_url
    ).json()

    plasma = pd.DataFrame(
        plasma_raw[1:],
        columns=plasma_raw[0]
    )

    plasma["density"] = pd.to_numeric(
        plasma["density"]
    )

    plasma["speed"] = pd.to_numeric(
        plasma["speed"]
    )

    latest_plasma = plasma.iloc[-1]

    proton_density = float(
        latest_plasma["density"]
    )

    solar_wind_speed = float(
        latest_plasma["speed"]
    )

    dynamic_pressure = (
        1.6726e-6
        * proton_density
        * solar_wind_speed**2
    )

    # ---------------------------
    # Magnetic Field
    # ---------------------------
    mag_url = (
        "https://services.swpc.noaa.gov/"
        "products/solar-wind/mag-1-day.json"
    )

    mag_raw = requests.get(
        mag_url
    ).json()

    mag = pd.DataFrame(
        mag_raw[1:],
        columns=mag_raw[0]
    )

    latest_mag = mag.iloc[-1]

    bx = float(latest_mag["bx_gsm"])
    by = float(latest_mag["by_gsm"])
    bz = float(latest_mag["bz_gsm"])
    bt = float(latest_mag["bt"])

    return [[
        flux_1,
        flux_3,
        flux_6,
        flux_12,
        solar_wind_speed,
        bz,
        proton_density,
        dynamic_pressure,
        bx,
        by,
        bt
    ]]

In [3]:
#from utils.data_fetcher import fetch_live_features

features = fetch_live_features()

print(features)
print(len(features[0]))

[[1081.5654296875, 1066.9954833984375, 1024.1815185546875, 1130.763427734375, 486.5, 1.31, 6.7, 2.6523607000450005, 2.23, -1.33, 2.94]]
11


In [6]:
import joblib

model_45m = joblib.load("../models/xgboost_45m_model.pkl")


prediction = model_45m.predict(features)

print(prediction)

[1119.8185]
